In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

# -----------------------------
# Paths (adjust if needed)
# -----------------------------
BASE_DIR = Path(".").resolve()   # run notebook from project root
INPUT_DIR = BASE_DIR / "annual_xslx"
MAPPINGS_DIR = BASE_DIR / "mappings"
OUT_DIR = BASE_DIR / "prof_components_extracted"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Your sheets are consistent: statement begins at Excel row 19 => skip first 18 rows
SKIPROWS = 18

# Only keep years <= 2024
MAX_YEAR = 2024

# Variable logic
PRIORITY_VARS = {"REVT", "XRD", "XINT", "BE", "MIB"}
SUM_VARS = {"COGS", "XSGA_COMPONENTS"}

ALL_VARS = ["REVT", "COGS", "XSGA_COMPONENTS", "XRD", "XINT", "BE", "MIB"]



In [ ]:

# -----------------------------
# Helpers
# -----------------------------
def parse_year(colname: str) -> Optional[int]:
    """
    Extract year from column name.
    Handles headers like '2024', 'FY2024', '31-12-2024', '2024-12-31', etc.
    Returns int year or None.
    """
    if colname is None:
        return None
    s = str(colname).strip()
    # Find a 4-digit year 19xx/20xx
    m = re.search(r"(19\d{2}|20\d{2})", s)
    if not m:
        return None
    return int(m.group(1))


def to_float(x):
    """Robust numeric parsing incl. European formats and thousand separators."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)

    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none"}:
        return np.nan

    # remove spaces / NBSP
    s = s.replace("\u00a0", "").replace(" ", "")

    # if both comma and dot exist, assume comma is thousands sep
    if "," in s and "." in s:
        s = s.replace(",", "")
    else:
        # if only comma exists, treat comma as decimal separator
        if "," in s and "." not in s:
            s = s.replace(",", ".")

    try:
        return float(s)
    except ValueError:
        return np.nan


def read_sheet_all_years(xlsx_path: Path, sheet_name: str) -> pd.DataFrame:
    """
    Read entire sheet starting at statement area. Keeps all year columns.
    """
    df = pd.read_excel(xlsx_path, sheet_name=sheet_name, skiprows=SKIPROWS)
    if df.empty or df.shape[1] == 0:
        return df

    label_col = df.columns[0]
    df[label_col] = df[label_col].astype(str).str.strip()
    df = df[df[label_col] != ""].copy()
    return df


def keep_year_cols_upto(df: pd.DataFrame, max_year: int) -> Tuple[pd.DataFrame, List[int]]:
    """
    Keeps only the year columns with parsed year <= max_year.
    Returns (df_filtered, years_sorted_desc_or_asc).
    """
    if df.empty:
        return df, []

    label_col = df.columns[0]
    year_cols = []
    years = []
    for c in df.columns[1:]:
        y = parse_year(c)
        if y is not None and y <= max_year:
            year_cols.append(c)
            years.append(y)

    # If multiple columns map to same year (rare), keep them all for now; we’ll aggregate later
    keep_cols = [label_col] + year_cols
    df2 = df[keep_cols].copy()

    # Standardize columns to year ints (may create duplicates)
    new_cols = [label_col] + [parse_year(c) for c in year_cols]
    df2.columns = new_cols

    # If duplicate year columns exist, collapse by taking first non-null across them (per row)
    # e.g., two columns both interpreted as 2024
    if len(set(new_cols[1:])) != len(new_cols[1:]):
        # group by year and take first non-null across duplicate columns
        label = df2.columns[0]
        out = df2[[label]].copy()
        for y in sorted(set(new_cols[1:])):
            cols_y = [c for c in df2.columns[1:] if c == y]
            block = df2[cols_y].applymap(to_float)
            out[y] = block.bfill(axis=1).iloc[:, 0]
        df2 = out

    years_sorted = sorted([c for c in df2.columns[1:] if isinstance(c, int)])
    return df2, years_sorted


def extract_label_series_with_duplicates(df_years: pd.DataFrame, row_label: str) -> pd.Series:
    """
    Extract a time series for a given row_label, handling duplicates:
    if multiple rows have same label, for each year take first non-null from top-most row,
    else next row, etc.
    """
    if df_years.empty:
        return pd.Series(dtype=float)

    label_col = df_years.columns[0]
    hits = df_years[df_years[label_col] == row_label]
    if hits.empty:
        return pd.Series(dtype=float)

    # numeric block: rows x years
    block = hits.drop(columns=[label_col]).applymap(to_float)

    # pick first non-null down the rows for each year (top-most wins)
    # bfill down rows then take first row
    picked = block.bfill(axis=0).iloc[0]
    picked.name = row_label
    return picked


def priority_across_labels(series_list: List[pd.Series]) -> pd.Series:
    """
    Given a list of label-series in priority order, pick first non-null per year.
    """
    if not series_list:
        return pd.Series(dtype=float)

    df = pd.concat(series_list, axis=1)  # years index
    out = df.bfill(axis=1).iloc[:, 0]
    return out


def sum_across_labels(series_list: List[pd.Series]) -> pd.Series:
    """
    Sum series across labels (align on years).
    """
    if not series_list:
        return pd.Series(dtype=float)
    df = pd.concat(series_list, axis=1)
    return df.sum(axis=1, min_count=1)


def load_mapping(mapping_path: Path) -> Dict:
    return json.loads(mapping_path.read_text(encoding="utf-8"))


In [ ]:


# -----------------------------
# Main extraction per firm
# -----------------------------
def extract_firm_components(xlsx_path: Path, mapping_path: Path, max_year: int = 2024) -> pd.DataFrame:
    mapping = load_mapping(mapping_path)
    variables = mapping.get("variables", [])

    # Cache sheet dfs (year-filtered) because we’ll read many labels per sheet
    sheet_cache: Dict[str, pd.DataFrame] = {}
    years_union = set()

    extracted: Dict[str, pd.Series] = {}

    for var_item in variables:
        var = var_item.get("variable", "")
        if var not in ALL_VARS:
            continue

        choices = var_item.get("final_choice", [])  # list of {sheet_name,row_label}

        # Empty final_choice -> return empty series for now (later fill with 0)
        if not choices:
            extracted[var] = pd.Series(dtype=float)
            continue

        # Extract a series for each chosen label (with duplicate-row handling)
        label_series_list: List[pd.Series] = []

        for ch in choices:
            sheet = ch["sheet_name"]
            label = ch["row_label"]

            if sheet not in sheet_cache:
                df_raw = read_sheet_all_years(xlsx_path, sheet)
                df_years, _ = keep_year_cols_upto(df_raw, max_year)
                sheet_cache[sheet] = df_years

            s = extract_label_series_with_duplicates(sheet_cache[sheet], label)
            if not s.empty:
                label_series_list.append(s)

        # Apply variable logic
        if var in PRIORITY_VARS:
            s_var = priority_across_labels(label_series_list)
        elif var in SUM_VARS:
            s_var = sum_across_labels(label_series_list)
        else:
            # fallback (shouldn't happen)
            s_var = sum_across_labels(label_series_list)

        extracted[var] = s_var
        years_union.update(list(s_var.index))

    # Build final DF over all years seen
    years_sorted = sorted([y for y in years_union if isinstance(y, int)])
    df_out = pd.DataFrame(index=years_sorted)
    df_out.index.name = "Year"

    for var in ALL_VARS:
        s = extracted.get(var, pd.Series(dtype=float))
        df_out[var] = s.reindex(years_sorted)

    # Fill missing series (e.g., final_choice=[]) with 0 as per your rule for later calculations
    # If you prefer leaving as NaN for now, change fillna(0.0) to keep NaN.
    df_out = df_out.fillna(0.0)

    return df_out



In [ ]:

# -----------------------------
# Run extraction for all firms
# -----------------------------
mapping_files = sorted(MAPPINGS_DIR.glob("*.json"))
print(f"Found {len(mapping_files)} mapping files in {MAPPINGS_DIR}")

n_ok, n_skip = 0, 0
for mp in mapping_files:
    firm_id = mp.stem
    xlsx_path = INPUT_DIR / f"{firm_id}.xlsx"
    if not xlsx_path.exists():
        n_skip += 1
        continue

    df = extract_firm_components(xlsx_path, mp, max_year=MAX_YEAR)
    out_path = OUT_DIR / f"{firm_id}.csv"
    df.to_csv(out_path)
    n_ok += 1

print(f"Done. Extracted {n_ok} firms. Skipped {n_skip} missing workbooks.")